# AutoML Tutorial: H2O AutoML
> **Run this notebook in Google Colab or Jupyter to learn about automated machine learning (AutoML) using H2O AutoML.**

## 1. Introduction
Automated Machine Learning (AutoML) simplifies the end-to-end ML workflow by:
- **Feature preprocessing**
- **Model selection**
- **Hyperparameter optimization**
- **Ensembling**

In this tutorial, we'll focus solely on **H2O AutoML**, a scalable AutoML framework supporting both regression and classification.


At the end, you'll complete an exercise applying AutoML.



## 2. Setup & Installation

In [ ]:
!pip install --quiet jedi
!pip install --quiet h2o
!pip install --quiet 'thinc<8.3.6'

## Regression on California Housing


In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 3. Regression Example: California Housing

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import h2o
from h2o.automl import H2OAutoML
from sklearn.metrics import mean_squared_error, r2_score

In [3]:
# Initialize H2O
h2o.init(max_mem_size="2G", nthreads=-1)

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; OpenJDK 64-Bit Server VM Temurin-25.0.4+7 (build 25.0.4+7-LTS, mixed mode, sharing)
  Starting server from C:\Users\HOME\anaconda3\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\HOME\AppData\Local\Temp\tmphd01b333
  JVM stdout: C:\Users\HOME\AppData\Local\Temp\tmphd01b333\h2o_HOME_started_from_python.out
  JVM stderr: C:\Users\HOME\AppData\Local\Temp\tmphd01b333\h2o_HOME_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,03 secs
H2O_cluster_timezone:,Asia/Karachi
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.11
H2O_cluster_version_age:,2 months and 6 days
H2O_cluster_name:,H2O_from_python_HOME_24cyq4
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.983 Gb
H2O_cluster_total_cores:,12
H2O_cluster_allowed_cores:,12
H2O_cluster_status:,"locked, healthy"


In [4]:
# Load data
data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target.rename('target')

In [5]:
# Create H2OFrame
df = h2o.H2OFrame(pd.concat([X, y], axis=1))

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [6]:
# Split into train/test
train, test = df.split_frame(ratios=[0.8], seed=42)

### 3.1 Run H2O AutoML for Regression

In [7]:
aml_reg = H2OAutoML(
    max_runtime_secs=300,
    max_models=20,
    seed=42,
    nfolds=5,
    project_name="california_regression"
)
aml_reg.train(x=X.columns.tolist(), y='target', training_frame=train)

AutoML progress: |
11:27:30.184: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: GBM_4_AutoML_1_20260728_112730


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    100                100                         282981                 10           10           10            55            449           220.93

ModelMetricsRegression: gbm
** Reported on train data. **

MSE: 0.07068300661149557
RMSE: 0.2658627589781908
MAE: 0.18120917114794807
RMSLE: 0.08210427216080109
Mean Residual Deviance: 0.07068300661149557

ModelMetricsRegression: gbm
** Reported on cross-validation data. **

MSE: 0.2034847118566311
RMSE: 0.4510927973894408
MAE: 0.29284213433945305
RMSLE: 0.13524205346487878
Mean Residual Deviance: 0.2034847118566311

Cross-Validation Metrics Summary: 
                        mean      sd          cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
----------------------  --------  ----------  ------------  ------------  ------------  ------------  ------------
aic                     nan       0           nan           nan           nan           nan           nan
loglikelihood           nan       0           nan           nan           nan           nan           nan
mae                     0.292841  0.00338635  0.296661      0.287396      0.293333      0.294019      0.292795
mean_residual_deviance  0.203505  0.00921118  0.217368      0.199415      0.205641      0.202794      0.192307
mse                     0.203505  0.00921118  0.217368      0.199415      0.205641      0.202794      0.192307
r2                      0.8475    0.0055502   0.840433      0.849944      0.845547      0.846212      0.855363
residual_deviance       0.203505  0.00921118  0.217368      0.199415      0.205641      0.202794      0.192307
rmse                    0.451024  0.0101703   0.466228      0.44656       0.453476      0.450326      0.438528
rmsle                   0.135246  0.00145376  0.136438      0.134214      0.136616      0.135689      0.133273

Scoring History: 
     timestamp            duration    number_of_trees    training_rmse        training_mae         training_deviance
---  -------------------  ----------  -----------------  -------------------  -------------------  -------------------
     2026-07-28 11:28:04  2.876 sec   0.0                1.155086044742462    0.9126682923030255   1.3342237707587847
     2026-07-28 11:28:05  3.012 sec   5.0                0.8050718297659779   0.6296027937088264   0.6481406510827397
     2026-07-28 11:28:05  3.067 sec   10.0               0.6022507490332858   0.45921175920543444  0.3627059647111538
     2026-07-28 11:28:05  3.114 sec   15.0               0.4899573038789042   0.36044403364806166  0.24005815962428487
     2026-07-28 11:28:05  3.154 sec   20.0               0.4331452215860772   0.30982096116895946  0.18761478298285192
     2026-07-28 11:28:05  3.202 sec   25.0               0.39683911573298336  0.2769064720926289   0.15748128377573614
     2026-07-28 11:28:05  3.239 sec   30.0               0.374152270539361    0.25627071575137306  0.13998992154975917
     2026-07-28 11:28:05  3.276 sec   35.0               0.3584411213603489   0.24324635480521686  0.12848003748206438
     2026-07-28 11:28:05  3.311 sec   40.0               0.3446381516088021   0.2327461465752868   0.11877545554433169
     2026-07-28 11:28:05  3.346 sec   45.0               0.3338572144190779   0.2247170105589096   0.11146063961966617
---  ---                  ---         ---                ---                  ---                  ---
     2026-07-28 11:28:05  3.418 sec   55.0               0.31870769879870814  0.21496977659685737  0.10157459727356806
     2026-07-28 11:28:05  3.455 sec   60.0               0

In [12]:
# Show leaderboard
df_leader_reg = aml_reg.leaderboard.as_data_frame()
print(df_leader_reg.head())

                                      model_id      rmse       mse       mae  \
0               GBM_4_AutoML_1_20260728_112730  0.451093  0.203485  0.292842   
1               GBM_3_AutoML_1_20260728_112730  0.451954  0.204262  0.297087   
2               GBM_2_AutoML_1_20260728_112730  0.456719  0.208592  0.301624   
3               GBM_1_AutoML_1_20260728_112730  0.458936  0.210623  0.303082   
4  GBM_grid_1_AutoML_1_20260728_112730_model_5  0.461007  0.212527  0.300461   

      rmsle  mean_residual_deviance  
0  0.135242                0.203485  
1  0.136120                0.204262  
2  0.137787                0.208592  
3  0.138777                0.210623  
4  0.138387                0.212527  


c:\Users\HOME\anaconda3\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [13]:
# Evaluate on test
perf_reg = aml_reg.leader.model_performance(test)
print(f"H2O Regression R²: {perf_reg.r2():.4f}")
print(f"H2O Regression RMSE: {perf_reg.rmse():.4f}")

H2O Regression R²: 0.8569
H2O Regression RMSE: 0.4346


## 4. Classification Example: Breast Cancer Dataset

In [14]:
from sklearn.datasets import load_breast_cancer

In [15]:
# Load and prepare dataset
data_cls = load_breast_cancer(as_frame=True)
Xc = data_cls.data
yc = data_cls.target.rename('target')

In [16]:
df_cls = h2o.H2OFrame(pd.concat([Xc, yc], axis=1))
train_cls, test_cls = df_cls.split_frame(ratios=[0.7], seed=42)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


### 4.1 Run H2O AutoML for Classification

In [17]:
aml_cls = H2OAutoML(
    max_runtime_secs=300,
    max_models=20,
    seed=42,
    nfolds=5,
    balance_classes=True,
    project_name="breast_cancer_classification"
)
aml_cls.train(x=Xc.columns.tolist(), y='target', training_frame=train_cls)

AutoML progress: |
11:40:46.251: AutoML: XGBoost is not available; skipping it.
11:40:46.285: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.


11:40:46.646: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

█
11:40:47.656: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

██
11:40:48.781: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical 

key,value
Stacking strategy,cross_validation
Number of base models (used / total),10/28
# GBM base models (used / total),8/25
# DRF base models (used / total),1/2
# GLM base models (used / total),1/1
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5
Metalearner fold_column,None
Custom metalearner hyperparameters,None


In [18]:
# Show leaderboard
df_leader_cls = aml_cls.leaderboard.as_data_frame()
print(df_leader_cls.head())

                                            model_id      rmse       mse  \
0  StackedEnsemble_AllModels_1_AutoML_2_20260728_...  0.181630  0.032989   
1  StackedEnsemble_BestOfFamily_1_AutoML_2_202607...  0.182539  0.033320   
2       GBM_grid_1_AutoML_2_20260728_114046_model_25  0.185965  0.034583   
3       GBM_grid_1_AutoML_2_20260728_114046_model_14  0.186716  0.034863   
4       GBM_grid_1_AutoML_2_20260728_114046_model_23  0.188528  0.035543   

        mae     rmsle  mean_residual_deviance  
0  0.097113  0.131007                0.032989  
1  0.104637  0.131654                0.033320  
2  0.096905  0.131893                0.034583  
3  0.089201  0.134120                0.034863  
4  0.090711  0.133354                0.035543  


c:\Users\HOME\anaconda3\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [19]:
# Evaluate on test
perf_cls = aml_cls.leader.model_performance(test_cls)
print(f"H2O Classification: {perf_cls}")

H2O Classification: ModelMetricsRegressionGLM: stackedensemble
** Reported on test data. **

MSE: 0.03556103996032552
RMSE: 0.18857635047991972
MAE: 0.09881519448380302
RMSLE: 0.12886513962124194
Mean Residual Deviance: 0.03556103996032552
R^2: 0.8421303082383419
Null degrees of freedom: 177
Residual degrees of freedom: 167
Null deviance: 40.43221198186829
Residual deviance: 6.329865112937943
AIC: -64.75570516846703


MSE (Mean Squared Error):
- Measures the average squared difference between predicted and actual values. Lower values indicate better performance.


RMSE (Root Mean Squared Error):
- The square root of MSE, providing an error metric in the same units as the target variable. Lower values are better.


MAE (Mean Absolute Error):
-Measures the average absolute difference between predicted and actual values. Lower values indicate better performance.


RMSLE (Root Mean Squared Logarithmic Error):
- Similar to RMSE but uses the logarithm of the values, making it more robust to outliers. Lower values are better.


Mean Residual Deviance:
- Measures the goodness of fit of the model. Lower values indicate a better fit.


R² (Coefficient of Determination):
- Represents the proportion of variance explained by the model. Values closer to 1 indicate better performance.


Null Degrees of Freedom:
- The number of observations minus 1.


Residual Degrees of Freedom:
- The number of observations minus the number of parameters estimated.


Null Deviance:
- The deviance of the null model (model with no predictors).


Residual Deviance:
- The deviance of the fitted model. Lower values indicate a better fit.


AIC (Akaike Information Criterion):
- A measure of model quality, balancing goodness of fit and model complexity. Lower values indicate a better model.

## 5. Interpreting Results
- **Leaderboard** displays model ranking by default metric.
- Use `model_performance` to compute custom metrics (RMSE, R², AUC, accuracy, etc.).
- Top models are automatically ensembled by H2O's Stacked Ensemble.

## 6. AutoML Best Practices
- **Set time and model limits** (`max_runtime_secs`, `max_models`) to control cost and runtime.
- **Use cross-validation** (`nfolds`) for robust performance estimates.
- **Balance classes** for imbalanced classification tasks.
- **Review variable importance** on the leader model: `aml.leader.varimp()`.
- **Save and deploy** the best model: `h2o.save_model(aml.leader, path='best_model')`.

## 7. Exercise: Custom Dataset AutoML

**Task:** Apply H2O AutoML to a custom dataset.

1. Load any tabular dataset (CSV or from `sklearn.datasets`).
2. Decide whether it's a regression or classification task.
3. Initialize H2O and convert to `H2OFrame`.
4. Split into appropriate train/test ratios.
5. Run `H2OAutoML` with:
   - `max_runtime_secs=300`
   - `max_models=15`
   - `nfolds=5`
   - `balance_classes=True` (if classification)
6. Display the leaderboard and evaluate on the test set using relevant metrics.
7. Save the leaderboard to a pandas DataFrame and export it as `leaderboard.csv`

In [26]:
#1. Load any tabular dataset (CSV or from sklearn.datasets).
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [27]:
#2. Decide whether it's a regression or classification task.
target = "Survived"

# Since Survived contains 0 (Did not survive) and 1 (Survived),
# this is a Binary Classification problem.

print(df[target].value_counts())

Survived
0    549
1    342
Name: count, dtype: int64


In [28]:
#3. Initialize H2O and convert to H2OFrame.
import h2o
from h2o.automl import H2OAutoML

# Start H2O cluster
h2o.init(max_mem_size="2G", nthreads=-1)

# Convert pandas DataFrame to H2OFrame
hf = h2o.H2OFrame(df)

# Convert target column to categorical for classification
hf[target] = hf[target].asfactor()

hf.head()

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,29 mins 37 secs
H2O_cluster_timezone:,Asia/Karachi
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.11
H2O_cluster_version_age:,2 months and 6 days
H2O_cluster_name:,H2O_from_python_HOME_24cyq4
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.791 Gb
H2O_cluster_total_cores:,12
H2O_cluster_allowed_cores:,12
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,nan,7.25,nan,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38,1,0,nan,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,nan,7.925,nan,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,113803,53.1,C123,S
5,0,3,"Allen, Mr. William Henry",male,35,0,0,373450,8.05,nan,S
6,0,3,"Moran, Mr. James",male,nan,0,0,330877,8.4583,nan,Q
7,0,1,"McCarthy, Mr. Timothy J",male,54,0,0,17463,51.8625,E46,S
8,0,3,"Palsson, Master. Gosta Leonard",male,2,3,1,349909,21.075,nan,S
9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27,0,2,347742,11.1333,nan,S
10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14,1,0,237736,30.0708,nan,C


In [29]:
#4. Split into appropriate train/test ratios.
train, test = hf.split_frame(ratios=[0.8], seed=42)

print("Train rows:", train.nrows)
print("Test rows:", test.nrows)

Train rows: 704
Test rows: 187


In [30]:
#5. Run H2OAutoML with:
#  max_runtime_secs=300
#  max_models=15
#  nfolds=5
#  balance_classes=True (if classification)
x = [col for col in hf.columns if col != target]

aml_clf = H2OAutoML(
    max_runtime_secs=300,
    max_models=15,
    nfolds=5,
    seed=42,
    balance_classes=True,
    project_name="Titanic_AutoML"
)

aml_clf.train(
    x=x,
    y=target,
    training_frame=train
)

AutoML progress: |█
11:56:51.590: AutoML: XGBoost is not available; skipping it.
11:56:51.628: _train param, Dropping bad and constant columns: [Name]
11:56:53.18: _train param, Dropping bad and constant columns: [Name]
11:56:54.315: _train param, Dropping bad and constant columns: [Name]

██
11:56:56.352: _train param, Dropping bad and constant columns: [Name]
11:56:57.466: _train param, Dropping bad and constant columns: [Name]
11:56:58.452: _train param, Dropping bad and constant columns: [Name]
11:56:59.196: _train param, Dropping bad and constant columns: [Name]
11:57:00.85: _train param, Dropping bad and constant columns: [Name]
11:57:00.609: _train param, Dropping bad and constant columns: [Name]

████████████████████████████████████████████████
11:59:09.337: _train param, Dropping unused columns: [Name]
11:59:10.887: _train param, Dropping unused columns: [Name]

████████████| (done) 100%


key,value
Stacking strategy,cross_validation
Number of base models (used / total),7/15
# GBM base models (used / total),2/8
# DeepLearning base models (used / total),3/4
# DRF base models (used / total),1/2
# GLM base models (used / total),1/1
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5
Metalearner fold_column,None


In [31]:
#6. Display the leaderboard
leaderboard = aml_clf.leaderboard
df_leaderboard = leaderboard.as_data_frame()
print(df_leaderboard.head())

#evaluate on the test set using relevant metrics.
best_model = aml_clf.leader
perf = best_model.model_performance(test)

print("Accuracy:", perf.accuracy())
print("AUC:", perf.auc())
print("LogLoss:", perf.logloss())
print("Confusion Matrix:")
print(perf.confusion_matrix())

c:\Users\HOME\anaconda3\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


                                            model_id       auc   logloss  \
0  StackedEnsemble_AllModels_1_AutoML_3_20260728_...  0.872095  0.412704   
1  StackedEnsemble_BestOfFamily_1_AutoML_3_202607...  0.867996  0.416365   
2                     GBM_4_AutoML_3_20260728_115651  0.860301  0.438410   
3        GBM_grid_1_AutoML_3_20260728_115651_model_3  0.856911  0.443704   
4  DeepLearning_grid_2_AutoML_3_20260728_115651_m...  0.854431  0.467030   

      aucpr  mean_per_class_error      rmse       mse  
0  0.848314              0.192819  0.359573  0.129293  
1  0.846999              0.197394  0.360712  0.130113  
2  0.828199              0.204106  0.372892  0.139048  
3  0.818088              0.184099  0.372609  0.138837  
4  0.818443              0.204784  0.376314  0.141612  
Accuracy: [[0.5565281261506084, 0.8128342245989305]]
AUC: 0.8296586059743954
LogLoss: 0.5436186154993445
Confusion Matrix:
Confusion Matrix (Act/Pred) for max f1 @ threshold = 0.5565281261506084
       0    

In [32]:
#7. Save the leaderboard to a pandas DataFrame 
df_leaderboard = aml_clf.leaderboard.as_data_frame()
df_leaderboard.head()

# export it as leaderboard.csv
df_leaderboard.to_csv("leaderboard.csv", index=False)
print("leaderboard.csv saved successfully.")

c:\Users\HOME\anaconda3\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


leaderboard.csv saved successfully.
